# GDS do circuito passivo

Este notebook só grava o GDS do filtro: cascata MODE, um grating coupler de entrada e quatro de saída, dentro do bloco de 1500 µm por 1800 µm.

Sem TiW, sem trilhas e sem DRC. A montagem na máscara e o DRC ficam em `main/main.py`.

Saída: `circuito-isa-jose-v1/saida/passivo.gds`. Rode a partir da raiz do repositório, com `uv run`.


In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import photonforge as pf
import siepic_forge as siepic


def find_circuito() -> Path:
    here = Path.cwd().resolve()
    for base in (here, *here.parents):
        if (base / "comum.py").is_file() and (base / "gds_passivo.ipynb").is_file():
            return base
        aninhado = base / "circuito-isa-jose-v1"
        if (aninhado / "comum.py").is_file() and (aninhado / "gds_passivo.ipynb").is_file():
            return aninhado
    raise FileNotFoundError("circuito-isa-jose-v1 não encontrado a partir do diretório atual")


CIRCUITO = find_circuito()
sys.path.insert(0, str(CIRCUITO))

import comum

sys.path.insert(0, str(comum.nano_root() / "src"))

import mzi_o4
import nanosoi

comum.SAIDA.mkdir(parents=True, exist_ok=True)
print(f"repositório {comum.repo_root()}")
print(f"saída {comum.SAIDA}")
print(f"photonforge {pf.__version__}")


repositório C:\Users\jose.roberto\Capacitacao\Estagio\repo_MZI_E_TPS\MZI-Estagio
saída C:\Users\jose.roberto\Capacitacao\Estagio\repo_MZI_E_TPS\MZI-Estagio\fluxo-passivo\saida
photonforge 1.5.4


In [2]:
mzi_o4.configure_pf()
cascata, _meta = mzi_o4.build_cascade_mode(root=comum.nano_root(), verbose=False)
gc = siepic.component(nanosoi.GC_NAME)
chip, _gmeta = mzi_o4.add_split_io_gcs(
    cascata, gc, pitch=nanosoi.PITCH_GC_UM, name="MZI_O4"
)
mzi_o4.add_circuit_model(chip)
mzi_o4.assert_no_fdtd(chip)
n_phys = mzi_o4.assert_physical(chip)
ports = set(chip.ports)
if ports != set(mzi_o4.FIB_PORTS):
    raise RuntimeError(f"filtro deve ter 1 entrada e 4 saídas, portas = {sorted(ports)}")

fx0, fy0, fx1, fy1 = comum.box(chip)
fw, fh = fx1 - fx0, fy1 - fy0
if fw > comum.AREA_W_UM + 1e-6 or fh > comum.AREA_H_UM + 1e-6:
    raise RuntimeError(
        f"filtro {fw:.1f} × {fh:.1f} µm não cabe em "
        f"{comum.AREA_W_UM:.0f} × {comum.AREA_H_UM:.0f} µm"
    )

dx = (comum.AREA_W_UM - fw) / 2.0 - fx0
dy = -fy0
top = pf.Component(comum.CELL_NAME)
top.add(
    nanosoi.FLOORPLAN_LAYER,
    pf.Rectangle((0.0, 0.0), (comum.AREA_W_UM, comum.AREA_H_UM)),
)
ref = top.add_reference(chip).translate((dx, dy))
for nome in mzi_o4.FIB_PORTS:
    top.add_port(ref[nome], port_name=nome)
comum.assert_no_layer6(top)

pf.write_layout(str(comum.PASSIVO_GDS), top, library_name="MZI_O4_PASSIVO")
x0, y0, x1, y1 = comum.box(top)
if (
    abs(x0) > 1e-3
    or abs(y0) > 1e-3
    or abs(x1 - comum.AREA_W_UM) > 1e-3
    or abs(y1 - comum.AREA_H_UM) > 1e-3
):
    raise RuntimeError(
        f"GDS {x1 - x0:.2f} × {y1 - y0:.2f} µm, "
        f"esperado {comum.AREA_W_UM:.0f} × {comum.AREA_H_UM:.0f}"
    )

print(f"GDS salvo em {comum.PASSIVO_GDS} ({comum.PASSIVO_GDS.stat().st_size / 1024:.1f} kB)")
print(f"bloco = {comum.AREA_W_UM:.0f} × {comum.AREA_H_UM:.0f} µm, célula {comum.CELL_NAME}")
print(
    f"filtro = {fw:.1f} × {fh:.1f} µm, "
    f"x = {fx0 + dx:.1f}–{fx1 + dx:.1f}, y = 0–{fy1 + dy:.1f}, "
    f"conexões físicas = {n_phys}"
)


MZI_O4_2estagios_liga_G1: (494.626, -2.750) -> (524.626, -381.351), L = 404.307 um
MZI_O4_2estagios_liga_G2: (494.626, 1.950) -> (524.626, 380.550), L = 404.306 um
folga vertical entre 200 GHz = 20.159 um
folga horizontal 100->200 GHz = 29.900 um
GDS salvo em C:\Users\jose.roberto\Capacitacao\Estagio\repo_MZI_E_TPS\MZI-Estagio\fluxo-passivo\saida\passivo.gds (94.6 kB)
bloco = 1500 × 1800 µm, célula mzi_o4_passivo
filtro = 1314.0 × 1503.6 µm, x = 93.0–1407.0, y = 0–1503.6, conexões físicas = 10


C:\Users\jose.roberto\AppData\Local\Temp\ipykernel_18248\1227372599.py:34: RuntimeWarning: The following components have been renamed in the layout because all names must be non-empty and unique: 'ebeam_y_1550_1', 'ebeam_dc_te1550_Lc6p59_1', 'bend__KSPMJOPB4VOMOK7UJEN6AAFZLYKZ7AQZXJGIZQ5SAUKLFFDPXNMA_1', 'bend__VI3NFRQQHDL4LGTLYHZXKFVE7FZ6PYS5DE62RKMUTC3Z3DV3J6OA_1', 'straight__G7TIRLNK5WJWLTUHCGD2Y2RNH5M7NHYT442UV2LP5F2EG72ZKXAQ_1', 'ebeam_dc_te1550_Lc6p24_1', 'ebeam_dc_te1550_Lc16p24_1', 'ebeam_dc_te1550_Lc10p16_1', 'ebeam_y_1550_2', 'ebeam_dc_te1550_Lc6p59_2', 'bend__KSPMJOPB4VOMOK7UJEN6AAFZLYKZ7AQZXJGIZQ5SAUKLFFDPXNMA_2', 'bend__VI3NFRQQHDL4LGTLYHZXKFVE7FZ6PYS5DE62RKMUTC3Z3DV3J6OA_2', 'straight__G7TIRLNK5WJWLTUHCGD2Y2RNH5M7NHYT442UV2LP5F2EG72ZKXAQ_2', 'ebeam_dc_te1550_Lc6p24_2', 'straight__S2IMFMKK6CDTYG3A32FO74IIL45AQN2SN43PSYJOC363L33CQWGQ_1', 'ebeam_dc_te1550_Lc16p24_2', 'ebeam_dc_te1550_Lc10p16_2', 'bend__KSPMJOPB4VOMOK7UJEN6AAFZLYKZ7AQZXJGIZQ5SAUKLFFDPXNMA_3', 'bend__VI3NFRQQH